### LST LMM

VARIABLE ROLES:
  - MAIN PREDICTORS:
    * Built-form characteristics (reldiff_built_up_ratio, reldiff_avg_building_height, reldiff_avg_building_footprint, diff_res_subclass_share_mfh_ab)
    * Environmental: NDVI_difference

  - CONTROL VARIABLES (CONFOUNDERS):
    * Temporal: years_since_construction_start
    * Spatial: latitude, longitude (geographic gradients)
    * Categorical: urban-rural classification, land-use type

Standardization: Z-score transformation (X - mean(X)) / std(X)
This allows a direct comparison of effect sizes between variables

In [ ]:

import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
from statsmodels.formula.api import mixedlm
import geopandas as gpd

warnings.filterwarnings('ignore', category=FutureWarning)

# ============================================================================
# Step 1: Load and prepare data
# ============================================================================

data_path = r"C:\Users\agz90fk\Documents\EO4CAM\3_Daten\Output\Masterarbeit\Analysis_v2\analysis_dataset.csv"
df = pd.read_csv(data_path)

# ============================================================================
# Step 2: Define variable structure with conceptual roles
# ============================================================================

outcome_var = 'difference_LST'

# Check for NDVI column name
ndvi_col = 'difference_NDVI'

# ============================================================================
# MAIN PREDICTORS: Built-form characteristics
# ============================================================================
built_form_vars = [
    'reldiff_built_up_ratio',
    'reldiff_avg_building_height',
    'reldiff_avg_building_footprint',
    'diff_res_subclass_share_mfh_ab'
]

# ============================================================================
# CONTROL VARIABLES: Environmental
# ============================================================================
environmental_control_vars = [
    ndvi_col  # Vegetation change as environmental control
]

# ============================================================================
# CONTROL VARIABLES: Temporal
# ============================================================================
temporal_control_vars = [
    'years_since_construction_start'
]

# ============================================================================
# CONTROL VARIABLES: Spatial (geographic gradients)
# ============================================================================
spatial_control_vars = [
    'lat',  # North-South gradient
    'lon'   # East-West gradient
]

# ============================================================================
# CONTROL VARIABLES: Categorical (urban-rural context, land cover)
# ============================================================================
categorical_control_vars = [
    'nhda_degurba_code',
    'nhda_sur_class_2021_short'
]

# Build comprehensive fixed_vars list (for formula)
fixed_vars = built_form_vars + environmental_control_vars + temporal_control_vars + spatial_control_vars

random_intercept_var = 'nhda_id'

# All variables needed for model
model_vars = [outcome_var] + fixed_vars + categorical_control_vars + [random_intercept_var]

# Prepare data
df_model = df[model_vars].copy().dropna()
df_model = df_model.reset_index(drop=True)

# ============================================================================
# Step 3: Outlier detection (EXCLUDING spatial controls)
# ============================================================================

# Only use outcome, built-form, environmental, and temporal for outlier detection
# DO NOT use lat/lon as they may be naturally variable
outlier_detection_vars = [outcome_var] + built_form_vars + environmental_control_vars + temporal_control_vars

z_scores = pd.DataFrame()
for var in outlier_detection_vars:
    z_scores[var] = np.abs(stats.zscore(df_model[var]))

outlier_mask = (z_scores.fillna(0) > 3).any(axis=1).values.astype(bool)
n_outliers = outlier_mask.sum()

# ============================================================================
# Step 4: Standardization
# ============================================================================

df_standardized = df_model.copy()
df_unstandardized = df_model.copy()

original_means = {}
original_stds = {}

for var in fixed_vars:
    original_means[var] = df_model[var].mean()
    original_stds[var] = df_model[var].std()

    # Standardize: (X - mean) / std
    df_standardized[var] = (df_model[var] - original_means[var]) / original_stds[var]

# Set reference categories for categorical variables
for df_temp in [df_standardized, df_unstandardized]:
    for cat_var in categorical_control_vars:
        df_temp[cat_var] = df_temp[cat_var].astype('category')

# ============================================================================
# Step 5: Split with/without outliers
# ============================================================================

df_standardized['has_outlier'] = outlier_mask
df_unstandardized['has_outlier'] = outlier_mask

df_std_with = df_standardized.copy()
df_std_without = df_standardized[~outlier_mask].copy()
df_unstd_with = df_unstandardized.copy()
df_unstd_without = df_unstandardized[~outlier_mask].copy()

for df_temp in [df_std_with, df_std_without, df_unstd_with, df_unstd_without]:
    df_temp = df_temp.drop(columns=['has_outlier'], errors='ignore').reset_index(drop=True)

# ============================================================================
# Step 6: Build model formula
# ============================================================================

# Build formula with organized variable structure
formula_parts = [f"{outcome_var} ~ "]

# Add built-form variables
formula_parts.append(" + ".join(built_form_vars))
formula_parts.append(" + ")

# Add control variables
formula_parts.append(" + ".join(environmental_control_vars))
formula_parts.append(" + ")
formula_parts.append(" + ".join(temporal_control_vars))
formula_parts.append(" + ")
formula_parts.append(" + ".join(spatial_control_vars))
formula_parts.append(" + ")

# Add categorical controls
formula_parts.append("C({})".format(") + C(".join(categorical_control_vars)))

formula = "".join(formula_parts)

# ============================================================================
# Step 7: Fit models (4 variants)
# ============================================================================

models_dict = {}

model1 = mixedlm(formula, data=df_unstd_with, groups=df_unstd_with[random_intercept_var])
models_dict['unstd_with'] = model1.fit(reml=True)

model2 = mixedlm(formula, data=df_unstd_without, groups=df_unstd_without[random_intercept_var])
models_dict['unstd_without'] = model2.fit(reml=True)

model3 = mixedlm(formula, data=df_std_with, groups=df_std_with[random_intercept_var])
models_dict['std_with'] = model3.fit(reml=True)

model4 = mixedlm(formula, data=df_std_without, groups=df_std_without[random_intercept_var])
models_dict['std_without'] = model4.fit(reml=True)

# ============================================================================
# Step 8: Extract fixed effects
# ============================================================================

def extract_fixed_effects(result):
    fe_index = result.fe_params.index
    coefs = result.fe_params[fe_index].values
    ses = result.bse[fe_index].values
    zvals = result.tvalues[fe_index].values
    pvals = result.pvalues[fe_index].values
    ci_lower = coefs - 1.96 * ses
    ci_upper = coefs + 1.96 * ses

    return pd.DataFrame({
        'Variable': fe_index,
        'Coefficient': coefs,
        'Std. Error': ses,
        'z-value': zvals,
        'p-value': pvals,
        '95% CI Lower': ci_lower,
        '95% CI Upper': ci_upper
    }).round(6)

results_dict = {}
for key, result in models_dict.items():
    results_dict[key] = extract_fixed_effects(result)

output_dir = Path(r"C:\Users\agz90fk\Documents\EO4CAM\3_Daten\Output\Masterarbeit\Analysis_v2\LMM")
output_dir.mkdir(parents=True, exist_ok=True)

# ============================================================================
# Step 9: Save results
# ============================================================================

# Unstandardized coefficients (original scale)
results_dict['unstd_with'].to_csv(
    output_dir / "1_LST_CONTROLS_unstd_WITH_outliers.csv", index=False
)

results_dict['unstd_without'].to_csv(
    output_dir / "2_LST_CONTROLS_unstd_WITHOUT_outliers.csv", index=False
)

# Standardized coefficients (1 SD change in X)
results_dict['std_with'].to_csv(
    output_dir / "3_LST_CONTROLS_std_WITH_outliers.csv", index=False
)

results_dict['std_without'].to_csv(
    output_dir / "4_LST_CONTROLS_std_WITHOUT_outliers.csv", index=False
)

# ============================================================================
# Step 10: Model fit comparison
# ============================================================================

model_fit_comparison = pd.DataFrame({
    'Model': ['WITH outliers', 'WITHOUT outliers'],
    'N Observations': [len(df_std_with), len(df_std_without)],
    'N Groups (NHDAs)': [df_std_with['nhda_id'].nunique(), df_std_without['nhda_id'].nunique()],
    'Log-Likelihood': [models_dict['std_with'].llf, models_dict['std_without'].llf],
    'AIC': [models_dict['std_with'].aic, models_dict['std_without'].aic],
    'BIC': [models_dict['std_with'].bic, models_dict['std_without'].bic]
})

model_fit_comparison.to_csv(
    output_dir / "5_LST_CONTROLS_model_fit.csv", index=False
)

# ============================================================================
# Step 12: Visualization of standardized LST coefficients
# ============================================================================

results_std_without = results_dict["std_without"].copy()

# ============================================================================
# Variable order
# ============================================================================

ordered_vars = (
    built_form_vars
    + environmental_control_vars
    + temporal_control_vars
    + spatial_control_vars
)

plot_results = results_std_without[
    results_std_without["Variable"].isin(ordered_vars)
].copy()

plot_results["var_order"] = plot_results["Variable"].map(
    {var: idx for idx, var in enumerate(ordered_vars)}
)

plot_results = (
    plot_results
    .sort_values("var_order")
    .reset_index(drop=True)
)

# ============================================================================
# Clean variable labels
# ============================================================================

label_map = {
    # Built-form characteristics
    "reldiff_built_up_ratio":
        r"Rel. $\Delta$ Built-up ratio",

    "reldiff_avg_building_height":
        r"Rel. $\Delta$ Building height",

    "reldiff_avg_building_footprint":
        r"Rel. $\Delta$ Building footprint",

    "diff_res_subclass_share_mfh_ab":
        r"$\Delta$ MFH-AB share",

    # Environmental characteristic
    "difference_NDVI":
        r"$\Delta$NDVI",

    # Temporal control
    "years_since_construction_start":
        "Years since construction",

    # Spatial controls
    "lat":
        "Latitude",

    "lon":
        "Longitude",

    # # DEGURBA controls
    # "C(nhda_degurba_code)[T.221]":
    #     "221 – Suburban/peri-urban",

    # "C(nhda_degurba_code)[T.222]":
    #     "222 – Semi-dense urban cluster",

    # "C(nhda_degurba_code)[T.223]":
    #     "223 – Dense urban cluster",

    # "C(nhda_degurba_code)[T.311]":
    #     "311 – Very low-density rural",

    # "C(nhda_degurba_code)[T.312]":
    #     "312 – Low-density rural",

    # "C(nhda_degurba_code)[T.313]":
    #     "313 – Rural cluster",

    # # Surrounding land-cover controls
    # "C(nhda_sur_class_2021_short)[T.2]":
    #     "Agricultural surroundings",

    # "C(nhda_sur_class_2021_short)[T.3]":
    #     "Forest/semi-natural surroundings",
}

plot_results["label"] = (
    plot_results["Variable"]
    .map(label_map)
    .fillna(plot_results["Variable"].astype(str))
)

# ============================================================================
# Assign conceptual variable groups
# ============================================================================

def get_role(var):
    if var in built_form_vars:
        return "Built-form characteristics"

    if var in environmental_control_vars:
        return "Environmental characteristic"

    if var in temporal_control_vars:
        return "Temporal control"

    if var in spatial_control_vars:
        return "Spatial controls"

    # if str(var).startswith("C(nhda_degurba_code)"):
    #     return "Urban-context controls"

    # if str(var).startswith("C(nhda_sur_class_2021_short)"):
    #     return "Land-cover controls"

    return "Other"


plot_results["role"] = plot_results["Variable"].apply(get_role)

# Same visual style as the ΔNDVI coefficient plot
role_colors = {
    "Built-form characteristics": "#ff595e",
    "Environmental characteristic": "#8ac926",
    "Temporal control": "#ffca3a",
    "Spatial controls": "#b5a6c9",
    # "Urban-context controls": "#7aa6c2",
    # "Land-cover controls": "#9c9c9c",
}

plot_results["color"] = (
    plot_results["role"]
    .map(role_colors)
    .fillna("#bdbdbd")
)

# Reverse order so the first conceptual group appears at the top
plot_results = plot_results.iloc[::-1].reset_index(drop=True)

# ============================================================================
# Create coefficient plot
# ============================================================================

fig, ax = plt.subplots(figsize=(9.5, 8.2))

y_pos = np.arange(len(plot_results))

# Confidence intervals and coefficient points
for i, row in plot_results.iterrows():

    ax.errorbar(
        row["Coefficient"],
        i,
        xerr=1.96 * row["Std. Error"],
        fmt="o",
        color=row["color"],
        ecolor=row["color"],
        alpha=0.75,
        markersize=9,
        markeredgecolor="black",
        markeredgewidth=0.8,
        elinewidth=2,
        capsize=4,
        zorder=3,
    )

# ============================================================================
# Axis labels with significance stars below variable labels
# ============================================================================

y_labels = []

for _, row in plot_results.iterrows():

    sig = ""

    if row["p-value"] < 0.001:
        sig = "\n$^{***}$"
    elif row["p-value"] < 0.01:
        sig = "\n$^{**}$"
    elif row["p-value"] < 0.05:
        sig = "\n$^{*}$"

    label = (
        str(row["label"])
        if pd.notna(row["label"])
        else str(row["Variable"])
    )

    y_labels.append(label + sig)

ax.set_yticks(y_pos)
ax.set_yticklabels(y_labels, fontsize=9)

ax.set_xlabel(
    "Standardized coefficient",
    fontsize=10,
)

# Zero-reference line
ax.axvline(
    0,
    color="black",
    linewidth=0.9,
    linestyle="--",
    zorder=1,
)

# Subtle grid
ax.grid(
    axis="x",
    alpha=0.25,
    linewidth=0.6,
)

ax.grid(
    axis="y",
    visible=False,
)

# ============================================================================
# Group separators
# ============================================================================

role_sequence = plot_results["role"].tolist()

for i in range(len(role_sequence) - 1):
    if role_sequence[i] != role_sequence[i + 1]:
        ax.axhline(
            i + 0.5,
            color="0.8",
            linewidth=0.7,
            zorder=0,
        )

# ============================================================================
# Legend
# ============================================================================

from matplotlib.lines import Line2D

legend_elements = [
    Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        label="Built-form",
        markerfacecolor=role_colors["Built-form characteristics"],
        markeredgecolor="black",
        alpha=0.75,
        markersize=10,
    ),

    Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        label="Environmental characteristic",
        markerfacecolor=role_colors["Environmental characteristic"],
        markeredgecolor="black",
        alpha=0.75,
        markersize=10,
    ),

    Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        label="Temporal control",
        markerfacecolor=role_colors["Temporal control"],
        markeredgecolor="black",
        alpha=0.75,
        markersize=10,
    ),

    Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        label="Spatial controls",
        markerfacecolor=role_colors["Spatial controls"],
        markeredgecolor="black",
        alpha=0.75,
        markersize=10,
    ),

#     Line2D(
#         [0],
#         [0],
#         marker="o",
#         color="w",
#         label="Urban-context controls",
#         markerfacecolor=role_colors["Urban-context controls"],
#         markeredgecolor="black",
#         alpha=0.75,
#         markersize=10,
#     ),

#     Line2D(
#         [0],
#         [0],
#         marker="o",
#         color="w",
#         label="Land-cover controls",
#         markerfacecolor=role_colors["Land-cover controls"],
#         markeredgecolor="black",
#         alpha=0.75,
#         markersize=10,
#     ),
]

ax.legend(
    handles=legend_elements,
    loc="lower left",
    bbox_to_anchor=(1.02, 0),
    fontsize=8,
    frameon=False,
    title="Variable group",
    title_fontsize=9,
)

# ============================================================================
# Title and styling
# ============================================================================

ax.set_title(
    r"Standardized coefficients for the $\Delta$LST model",
    fontsize=11,
    fontweight="bold",
    pad=10,
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.subplots_adjust(
    left=0.33,
    right=0.77,
    top=0.92,
    bottom=0.10,
)

# ============================================================================
# Export
# ============================================================================

coef_plot = output_dir / "06_DELTA_LST_coefficient_plot_clean.jpg"

plt.savefig(
    coef_plot,
    dpi=300,
    bbox_inches="tight",
)

plt.close()

# Model diagnostics

In [ ]:
# ============================================================================
# Step 13: Model diagnostics for the ΔLST LMM
# Residuals versus fitted values and normal Q-Q plot
# ============================================================================
print("\n" + "=" * 80)
print("STEP 13: MODEL DIAGNOSTICS FOR ΔLST LMM")
print("=" * 80)

import statsmodels.api as sm

# Primary model: unstandardized model without outliers
diagnostic_model = models_dict["unstd_without"]

# Extract conditional fitted values and residuals
fitted_values = np.asarray(
    diagnostic_model.fittedvalues,
    dtype=float
)

residuals = np.asarray(
    diagnostic_model.resid,
    dtype=float
)

# Remove non-finite values defensively
valid_mask = (
    np.isfinite(fitted_values)
    & np.isfinite(residuals)
)

fitted_values = fitted_values[valid_mask]
residuals = residuals[valid_mask]

# Standardize residuals
residual_mean = np.mean(residuals)
residual_sd = np.std(residuals, ddof=1)

if not np.isfinite(residual_sd) or residual_sd == 0:
    raise ValueError(
        "Residual standard deviation is zero or non-finite."
    )

standardized_residuals = (
    residuals - residual_mean
) / residual_sd


# ============================================================================
# 1. Residuals versus fitted values
# ============================================================================

fig, ax = plt.subplots(figsize=(7.2, 5.2))

ax.scatter(
    fitted_values,
    standardized_residuals,
    s=18,
    alpha=0.25,
    edgecolors="none",
    color = "darkgrey"
)

# Horizontal reference line
ax.axhline(
    y=0,
    color="black",
    linestyle="--",
    linewidth=1
)

# LOWESS smoother
lowess_values = sm.nonparametric.lowess(
    endog=standardized_residuals,
    exog=fitted_values,
    frac=0.30,
    it=3,
    return_sorted=True
)

ax.plot(
    lowess_values[:, 0],
    lowess_values[:, 1],
    linewidth=2
)

ax.set_xlabel(
    r"Fitted $\Delta$LST values",
    fontsize=10
)

ax.set_ylabel(
    "Standardized residuals",
    fontsize=10
)

ax.set_title(
    r"Residuals versus fitted values for the $\Delta$LST LMM",
    fontsize=11,
    fontweight="bold",
    pad=10
)

ax.grid(
    axis="both",
    alpha=0.20,
    linewidth=0.6
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

residual_plot = (
    output_dir
    / "DELTA_LST_residuals_vs_fitted_without_outliers.jpg"
)

plt.savefig(
    residual_plot,
    dpi=300,
    bbox_inches="tight"
)

print(
    f"✓ Residual-versus-fitted plot saved: "
    f"{residual_plot}"
)

plt.close()


# ============================================================================
# 2. Normal Q-Q plot
# ============================================================================

fig, ax = plt.subplots(figsize=(6.2, 5.2))

sm.qqplot(
    standardized_residuals,
    line="45",
    fit=True,
    ax=ax,
    marker="o",
    markerfacecolor="none",
    markeredgecolor="black",
    alpha=0.30
)

ax.set_xlabel(
    "Theoretical quantiles",
    fontsize=10
)

ax.set_ylabel(
    "Standardized residual quantiles",
    fontsize=10
)

ax.set_title(
    r"Normal Q--Q plot for the $\Delta$LST LMM",
    fontsize=11,
    fontweight="bold",
    pad=10
)

ax.grid(
    axis="both",
    alpha=0.20,
    linewidth=0.6
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

qq_plot = (
    output_dir
    / "DELTA_LST_normal_QQ_without_outliers.jpg"
)

plt.savefig(
    qq_plot,
    dpi=300,
    bbox_inches="tight"
)

print(
    f"✓ Normal Q-Q plot saved: "
    f"{qq_plot}"
)

plt.close()


# ============================================================================
# 3. Combined diagnostic figure
# ============================================================================

fig, axes = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=(11.5, 4.8)
)

# Panel A: Residuals versus fitted values
axes[0].scatter(
    fitted_values,
    standardized_residuals,
    s=18,
    alpha=0.25,
    edgecolors="none"
)

axes[0].axhline(
    y=0,
    color="black",
    linestyle="--",
    linewidth=1
)

axes[0].plot(
    lowess_values[:, 0],
    lowess_values[:, 1],
    linewidth=2
)

axes[0].set_xlabel(
    r"Fitted $\Delta$LST values"
)

axes[0].set_ylabel(
    "Standardized residuals"
)

axes[0].set_title(
    "a) Residuals versus fitted values",
    fontsize=10
)

# Panel B: Normal Q-Q plot
sm.qqplot(
    standardized_residuals,
    line="45",
    fit=True,
    ax=axes[1],
    marker="o",
    markerfacecolor="none",
    markeredgecolor="black",
    alpha=0.30
)

axes[1].set_xlabel(
    "Theoretical quantiles"
)

axes[1].set_ylabel(
    "Standardized residual quantiles"
)

axes[1].set_title(
    "b) Normal Q--Q plot",
    fontsize=10
)

# Common styling
for current_ax in axes:

    current_ax.grid(
        alpha=0.20,
        linewidth=0.6
    )

    current_ax.spines["top"].set_visible(False)
    current_ax.spines["right"].set_visible(False)

fig.suptitle(
    r"Diagnostic plots for the $\Delta$LST LMM",
    fontsize=12,
    fontweight="bold",
    y=1.02
)

plt.tight_layout()

combined_plot = (
    output_dir
    / "30_DELTA_LST_model_diagnostics_without_outliers.jpg"
)

plt.savefig(
    combined_plot,
    dpi=300,
    bbox_inches="tight"
)

print(
    f"✓ Combined diagnostic plot saved: "
    f"{combined_plot}"
)

plt.close()